In [2]:
import pandas as pd
import numpy as np
import belo_horizonte_real_estate_market.functions.match_info as match_info
import belo_horizonte_real_estate_market.functions.sources as dicts

In [4]:
pd.set_option('display.max_columns', None)

In [284]:
df_atividades_economicas = pd.read_parquet(path = "../data/raw_atividades_economicas.parquet")\
.assign(data_inicio_atividade = lambda df: df["data_inicio_atividade"].str.replace(r"/", "").str.replace("-", ""))\
.assign(cnae_principal = lambda df: df["cnae_principal"].str.replace("\.0", "", regex = True))\
.assign(cnae_secundarias = lambda df: df["cnae_secundarias"].str.replace("nan", "").str.split(", "))\
.assign(nome_fantasia = lambda df: df["nome_fantasia"].str.replace("nan", ""))\
.assign(complemento = lambda df: df["complemento"].str.replace("nan", ""))\
.assign(desc_logradouro = lambda df: df["desc_logradouro"].map(dicts.DATASETS["TIPO_LOGRADOURO"]))


In [285]:
df_empresarios_individuais = df_atividades_economicas\
.query("natureza_juridica.str.contains('EMPRES.RIO \(INDIVIDUAL\)')")\
[["cnpj", "nome", "nome_fantasia", "data_inicio_atividade", "nome_regional", "cnae_principal", "cnae_secundarias", "descricao_cnae", "porte_empresa", "area_utilizada", "ind_simples", "ind_mei", 
  "tipo_unidade", "forma_atuacao", "desc_logradouro", "nome_logradouro", "numero_imovel", "complemento", "nome_bairro", "geometria"]]\
.assign(ind_id = lambda df: df["nome"].str.contains(r"[0-9]{2}\.[0-9]{3}\.[0-9]{3}|[0-9]{3}.[0-9]{3}.[0-9]{3}-[0-9]{2}|[0-9]{11}|[0-9]{9}|[0-9]{3} [0-9]{3} [0-9]{3}|CPF [0-9]|CPF\:").astype(int).astype(str))\
.assign(ind_id = lambda df: df["ind_simples"] + df["ind_mei"] + df["ind_id"])\
.assign(tp_nome = lambda df: df["nome"].str.replace(r"[.\-:()\,]", "", regex = True))\
.assign(tp_nome = lambda df: df["tp_nome"].str.replace(r"[\x00-\x1F\x7F]", "", regex = True))\
.assign(tp_nome = lambda df: df["tp_nome"].str.replace(r"(?<=\d)\s+(?=\d)", "", regex = True))\
.assign(tp_nome = lambda df: df["tp_nome"].str.strip())\
.assign(tp_nome = lambda df: df["tp_nome"].str.replace("(^.+[A-Z])([0-9].+$)", "\\1 \\2", regex = True))\
.assign(tp_ = lambda df: np.where(df["ind_id"].apply(lambda x: x[2] == "1"), 0, -1))\
.assign(tp_ = lambda df: np.where(df["tp_nome"].str.contains("^.+?CPF\s+[0-9]+$").astype("int") == 1, 1, df["tp_"]))\
.assign(tp_ = lambda df: np.where((df["tp_nome"].str.contains(".+[A-Z]CPF\s+[0-9]+")) & (df["tp_"] == 1), 8, df["tp_"]))\
.assign(tp_ = lambda df: np.where((df["tp_nome"].str.contains("^[0-9]+.+$")) & (df["tp_"] == 0), 2, df["tp_"]))\
.assign(tp_ = lambda df: np.where((df["tp_nome"].str.contains("^.+?\s+[0-9]+$")) & (df["tp_"] == 0), 3, df["tp_"]))\
.assign(tp_ = lambda df: np.where((df["tp_nome"].str.contains("^.+?CPF[0-9]+$")) & (df["tp_"] == 0), 4, df["tp_"]))\
.assign(tp_ = lambda df: np.where((df["tp_nome"].str.contains("^.+?CPF\s+[0-9]+\s+[A-Z]+")) & (df["tp_"] == 0), 5, df["tp_"]))\
.assign(tp_ = lambda df: np.where((df["tp_nome"].str.contains("^.+?[0-9]+\s+[A-Z]+")) & (df["tp_"] == 0), 6, df["tp_"]))\
.assign(tp_ = lambda df: np.where((df["tp_nome"].str.contains("^.+?CPF\s+[0-9]+[A-Z]+")) & (df["tp_"] == 0), 7, df["tp_"]))\
.assign(tp_list = lambda df: np.where(df["tp_"] == 1, df["tp_nome"].str.split(" CPF "), df["tp_nome"]))\
.assign(tp_list = lambda df: np.where(df["tp_"] == 2, df["tp_nome"].str.replace("([0-9]+)(\s+)(.+$)", "\\3;\\1", regex = True).str.split(";"), df["tp_list"]))\
.assign(tp_list = lambda df: np.where(df["tp_"] == 3, df["tp_nome"].str.replace("^(.+)(\s+)([0-9]+$)", "\\1;\\3", regex = True).str.split(";"), df["tp_list"]))\
.assign(tp_list = lambda df: np.where(df["tp_"] == 4, df["tp_nome"].str.split("CPF"), df["tp_list"]))\
.assign(tp_list = lambda df: np.where(df["tp_"] == 5, df["tp_nome"].str.replace("^(.+)( CPF )([0-9]+)(\s+?[A-Z]+)", "\\1;\\3", regex = True).str.split(";"), df["tp_list"]))\
.assign(tp_list = lambda df: np.where(df["tp_"] == 6, df["tp_nome"].str.replace("^(.+)(\s+)([0-9]+)(\s+?[A-Z]+)", "\\1;\\3", regex = True).str.split(";"), df["tp_list"]))\
.assign(tp_list = lambda df: np.where(df["tp_"] == 7, df["tp_nome"].str.replace("^(.+)(\s+CPF\s+)([0-9]+)(.+)", "\\1;\\3", regex = True).str.split(";"), df["tp_list"]))\
.assign(tp_list = lambda df: np.where(df["tp_"] == 8, df["tp_nome"].str.replace("^(.+)(CPF\s+)([0-9]+)", "\\1;\\3", regex = True).str.split(";"), df["tp_list"]))\
.assign(pessoa = lambda df: np.where(df["tp_"] == -1, df["tp_nome"], df["tp_list"].apply(lambda x: x[0])))\
.assign(cpf = lambda df: np.where(df["tp_"] == -1, "", df["tp_list"].apply(lambda x: x[1])))\
.drop(columns = ["tp_nome", "ind_id", "tp_nome", "tp_"])


In [286]:
df_empresarios_individuais["forma_atuacao"].value_counts()

forma_atuacao
ESTABELECIMENTO FIXO                                                                                                        166808
PORTA A PORTA, POSTOS MÓVEIS OU POR AMBULANTES                                                                               84898
INTERNET                                                                                                                     65078
EM LOCAL FIXO FORA DE LOJA                                                                                                   34522
ESTABELECIMENTO FIXO, INTERNET                                                                                               33198
                                                                                                                             ...  
ATIVIDADES DESENVOLVIDAS FORA DO ESTABELECIMENTO, EM LOCAL FIXO FORA DE LOJA, ESTABELECIMENTO FIXO, MÁQUINAS AUTOMÁTICAS         1
CORREIO, EM LOCAL FIXO FORA DE LOJA, MÁQUINAS AUTOMÁTICAS, TELEVENDAS

In [287]:
df_empresarios_individuais.query("ind_simples == 'N'")

,cnpj,nome,nome_fantasia,data_inicio_atividade,nome_regional,cnae_principal,cnae_secundarias,descricao_cnae,porte_empresa,area_utilizada,ind_simples,ind_mei,tipo_unidade,forma_atuacao,desc_logradouro,nome_logradouro,numero_imovel,complemento,nome_bairro,geometria,tp_list,pessoa,cpf
index,,,,,,,,,,,,,,,,,,,,,,,
43,70986948000143,ROBSON SALES HENEDINO,,01041993,NORDESTE,4520001,"[4520002, 4520003, 4520004]",SERVICOS DE MANUTENCAO E REPARACAO MECANICA DE...,MICROEMPRESA - ME,150.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,DOUTOR BENJAMIM MOSS,319,,UNIAO,POINT (613033.200778877 7800638.00374008),ROBSON SALES HENEDINO,ROBSON SALES HENEDINO,
109,18716712000177,ANTONIO HENRIQUES DA SILVA,BILHARES CEU AZUL,02011975,VENDA NOVA,3240002,[3240003],"FABRICACAO DE MESAS DE BILHAR, DE SINUCA E ACE...",MICROEMPRESA - ME,56.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,AVELINO GIAROLA,475,LOJA 1,CEU AZUL,POINT (604815.573078683 7807629.62889447),ANTONIO HENRIQUES DA SILVA,ANTONIO HENRIQUES DA SILVA,
119,71065932000160,EDUARDO FERREIRA DA SILVA,SUPERMERCADO SAO GERALDO,10051993,LESTE,4711302,[],"COMERCIO VAREJISTA DE MERCADORIAS EM GERAL, CO...",MICROEMPRESA - ME,417.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,PRACA,SANTUARIO SAO GERALDO,151,,SAO GERALDO,POINT (615316.549277029 7799479.17961538),EDUARDO FERREIRA DA SILVA,EDUARDO FERREIRA DA SILVA,
193,42994152000105,FABIO KALKS KOSCKY,BAR DO FABIO,01011993,NOROESTE,5611203,[],SORVETERIA,MICROEMPRESA - ME,23.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,AVENIDA,TRINTA E UM DE MARCO,390,A,DOM CABRAL,POINT (605097.201696115 7796886.79798928),FABIO KALKS KOSCKY,FABIO KALKS KOSCKY,
201,23951874000120,DALMO BECATTINI FILHO,MERCEARIA ESQUININHA,01031993,NOROESTE,4729699,[4712100],COMERCIO VAREJISTA DE PRODUTOS ALIMENTICIOS EM...,MICROEMPRESA - ME,50.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,CANTAGALO,821,,APARECIDA,POINT (609719.880503579 7800057.78886091),DALMO BECATTINI FILHO,DALMO BECATTINI FILHO,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1453286,11267429000166,ITAMERES MOREIRA SALOMAO DO SANTO,,02092021,nan,9602502,"[4772500, 9602502]",ATIVIDADES DE ESTÉTICA E OUTROS SERVIÇOS DE CU...,EMPRESA DE PEQUENO PORTE,81.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,SAO PAULO,1966,"SALA:401,",LOURDES,POINT (610590.27 7795834.87),ITAMERES MOREIRA SALOMAO DO SANTO,ITAMERES MOREIRA SALOMAO DO SANTO,
1463926,18946854000120,MARCIA ELISA GUEDES CONCEICAO 02691869644,,25092013,nan,1412601,"[1412601, 3299099]","CONFECCAO DE PECAS DO VESTUARIO, EXCETO ROUPAS...",MICROEMPRESA - ME,10.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,MARY APARECIDA,186,,JARDIM SAO JOSE,POINT (605177.15 7799017.97),"[MARCIA ELISA GUEDES CONCEICAO, 02691869644]",MARCIA ELISA GUEDES CONCEICAO,02691869644
1488477,13647909000188,DUILIO DUTRA FILHO,,09012024,nan,4221905,[4221905],MANUTENCAO DE ESTACOES E REDES DE TELECOMUNICA...,MICROEMPRESA - ME,10.0,N,N,UNIDADE PRODUTIVA,"PORTA A PORTA, POSTOS MÓVEIS OU POR AMBULANTES",RUA,RIO GRANDE DO NORTE,1435,SALA 708,SAVASSI,POINT (611699.07 7795022.80),DUILIO DUTRA FILHO,DUILIO DUTRA FILHO,


In [291]:
df_atividades_economicas\
.query("cnae_principal == '8112500'")["natureza_juridica"].value_counts()

natureza_juridica
CONDOMÍNIO EDILÍCIO                   26165
ASSOCIAÇÃO PRIVADA                       42
SOCIEDADE EMPRESÁRIA LIMITADA            12
CONSÓRCIO DE SOCIEDADES                   2
SOCIEDADE SIMPLES LIMITADA                2
SOCIEDADE EM CONTA DE PARTICIPAÇÃO        1
COMUNIDADE INDÍGENA                       1
Name: count, dtype: int64

In [301]:
df_condominios_cnpjs = df_atividades_economicas\
.query("natureza_juridica == 'CONDOMÍNIO EDILÍCIO'")\
[["cnpj", "nome", "nome_fantasia", "data_inicio_atividade", "nome_regional", "cnae_principal", "cnae_secundarias", "descricao_cnae", "porte_empresa", "area_utilizada", "ind_simples", "ind_mei", 
  "tipo_unidade", "forma_atuacao", "desc_logradouro", "nome_logradouro", "numero_imovel", "complemento", "nome_bairro", "geometria"]]

In [336]:
df_condominios_cnpjs

,cnpj,nome,nome_fantasia,data_inicio_atividade,nome_regional,cnae_principal,cnae_secundarias,descricao_cnae,porte_empresa,area_utilizada,ind_simples,ind_mei,tipo_unidade,forma_atuacao,desc_logradouro,nome_logradouro,numero_imovel,complemento,nome_bairro,geometria
index,,,,,,,,,,,,,,,,,,,,
2411,65162380000106,CONDOMINIO MINAS SHOPPING,,25091991,NORDESTE,8112500,"[5223100, 9329899]",CONDOMINIOS PREDIAIS,DEMAIS,16895.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,AVENIDA,CRISTIANO MACHADO,4000,,UNIAO,POINT (612416.154541622 7801830.90704951)
10211,19794254000157,CONDOMINIO GALERIA OUVIDOR,,01102000,CENTRO-SUL,8112500,[9609299],CONDOMINIOS PREDIAIS,DEMAIS,80.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,CURITIBA,715,,CENTRO,POINT (610812.067175938 7797128.25813863)
12297,4782680000101,CONDOMINIO DO EDIFICIO AFONSO PENA FLAT SERVICE,,27042000,CENTRO-SUL,8112500,[],CONDOMINIOS PREDIAIS,DEMAIS,100.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,AVENIDA,AFONSO PENA,3761,,SERRA,POINT (612655.663545189 7794450.97968948)
20991,65172074000150,CONDOMINIO SOLDADO WILSON TRINDADE,,19021992,NOROESTE,8112500,[],CONDOMINIOS PREDIAIS,DEMAIS,1.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,ANTONIO PEIXOTO,91,,COQUEIROS,POINT (602153.435914513 7799276.76081714)
21916,9198026000160,CONDOMINIO EDIFICIO ITUIUTABA,,19072007,OESTE,8112500,[],CONDOMINIOS PREDIAIS,DEMAIS,643.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,CONSELHEIRO JOAQUIM CAETANO,223,,NOVA GRANADA,POINT (607785.270531208 7794969.23767373)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1367953,63209545000122,CONDOMINIO PAMPULHA,,04092025,nan,8112500,[8112500],CONDOMINIOS PREDIAIS,DEMAIS,10.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,AVENIDA,FLEMING,510,,OURO PRETO,POINT (606165.02 7802623.63)
1368324,25467952000140,CONDOMINIO DO EDIFICIO PALERMO,,18072025,nan,8112500,[8112500],CONDOMINIOS PREDIAIS,DEMAIS,10.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,PROFESSOR NELSON DE SENA,115,,AEROPORTO,POINT (609113.11 7804142.15)
1412057,4133916000170,CONDOMINIO DO RESIDENCIAL OHIO - BLOCO 04,,08112000,nan,8112500,[8112500],CONDOMINIOS PREDIAIS,DEMAIS,10.0,N,N,UNIDADE PRODUTIVA,nan,RUA,HENRIQUE FURTADO PORTUGAL,180,BLOCO 04,BURITIS,POINT (608195.15 7791098.83)


In [ ]:
pd.DataFrame(df_condominios_cnpjs["nome"].str.split(" ").to_list())\
.stack().value_counts().reset_index().sort_values("index")\
["index"].to_list()[2000:3000]

# C0NDOMINIO, CNDOMINIO, CODOMINIO, COND, COND. 'CONDIMINIO',
 'CONDMINIO',
 'CONDOIMINIO',
 'CONDOMIINIO',
 'CONDOMIIO',
 'CONDOMIMIO',
 'CONDOMIMNIO',
 'CONDOMIN',
 'CONDOMINI0',
 'CONDOMINIIO',
 'CONDOMININIO',
 'CONDOMININO',
 'CONDOMINIO', CONDOMINJIO, 'CONDOMINMIO',
 'CONDOMINNIO',
 'CONDOMINO',
 'CONDOMINOS',
 'CONDOMIO',
 'CONDOMIONIO',
 'CONDOMN',
 'CONDOMNIO',
 'CONDOMONIO',
 'CONDOMÌNIO',
 'CONDOMÍNIO',
 'CONDONIMIO',
 'CONDONINIO', 'CONMDOMINIO',
 'CONODMINIO',
 'CONOMINIO',

['CIO',
 'CIOLETTI',
 'CIOTTO',
 'CIPO',
 'CIPRESTE',
 'CIPRIANI',
 'CIPRIANO',
 'CIPROS',
 'CIRA',
 'CIRCE',
 'CIRELI',
 'CIRIACO',
 'CIRILO',
 'CIRIUS',
 'CIRO',
 'CIRRUS',
 'CISA',
 'CISNE',
 'CITE',
 'CITIBANK',
 'CITINUS',
 'CITRINO',
 'CITTA',
 'CITY',
 'CIVENA',
 'CIVETTA',
 'CIVIL',
 'CLA',
 'CLAIR',
 'CLAIRE',
 'CLARA',
 'CLARAS',
 'CLARET',
 "CLARET''",
 'CLARICE',
 'CLARINDA',
 'CLARION',
 'CLARISSA',
 'CLARO',
 'CLAROS',
 'CLASS',
 'CLASSIC',
 'CLASSICA',
 'CLAUDE',
 'CLAUDEL',
 'CLAUDIA',
 'CLAUDIANO',
 'CLAUDIENSE',
 'CLAUDINA',
 'CLAUDINO',
 'CLAUDIO',
 'CLAUDIOMAR',
 'CLAUDIONOR',
 'CLAUDOVALDO',
 'CLAUSIO',
 'CLAVE',
 'CLEIDE',
 'CLELIA',
 'CLEMENT',
 'CLEMENTE',
 'CLEMENTINO',
 'CLEONICE',
 'CLERMOND',
 'CLERMONT',
 'CLERY',
 'CLEUZA',
 'CLEVELAND',
 'CLIMA',
 'CLINICA',
 'CLOCO',
 'CLODOVEL',
 'CLODOVEU',
 'CLORIS',
 'CLOTILDE',
 'CLOVIS',
 'CLUB',
 'CLUBE',
 'CNDOMINIO',
 'COBALTO',
 'COBAP',
 'COBERIO',
 'COBRA',
 'COBUCCI',
 'COCAIS',
 'COD',
 'CODO',
 'CODOMINIO'

In [ ]:
df_condominios_cnpjs\
.assign(nome_condominio = lambda df: df["nome"].str.replace("[\'\-]", "", regex = True).str.replace("\s+", " ", regex = True).str.strip())\
.assign(nome_condominio = lambda df: df["nome_condominio"].str.replace("CONDOMINIO ", "", regex = True))\
.sort_values("nome_condominio")\
["nome_condominio"].to_list()
 

['2300 RIO DE JANEIRO',
 'ADMINISTRACAO DE COMPOSSUIDORES DO PROPRIO NACIONAL RESIDENCIAL EDIFICIO GENERAL ANTONIO BANDEIRA',
 'AIMORES CONDOMINIO DO EDIFICIO AIMORES',
 'ALENQUER ALCOE RESIDENCE',
 'ALFA CONDOMINIO EMPRESARIAL',
 'ALMA RESIDENCE',
 'ALTA VISTA DO PALMARES RESIDENCIAL',
 'ALTO GUTIERREZ CONDOMINIO BOUTIQUE',
 'ALTO HORIZONTE RESIDENCIAL',
 'APART HOTEL MAURILIO LAGES',
 'APARTHOTEL MAURILIO LAGES',
 'ASSOC DE MORADORES DO FAZENDA DA SERRA AMFS',
 'ASSOC.PROP.APTOS BLB COND.CONJ.HAB.DR.RODRIGO M.FRANCO',
 'ASSOCIACAO CONDOMINIO RESIDENCIAL PARQUET',
 'ASSOCIACAO CONDOMINIO RESIDENCIAL PARQUET',
 'ASSOCIACAO PROPRIETARIOS APTS BL B CONDOMINIO EDIFICIO CONJ. HAB. DR RODRIGO MELO FRANCO',
 'BE EASY STUDIOS RIVA',
 'BEATRIZ SIMAO RESIDENCE',
 'BELLA TOSCANA',
 'BELLAGIO ALCOE RESIDENCE',
 'BETANIA PARK RESIDENCIAS',
 'BH SQUARE CENTER',
 'BHDECOR',
 'BL II COND RESIDENCIAL ROSA ANDRADE SILVA',
 'BL III COND RESIDENCIAL ROSA ANDRADE SILVA',
 'BL IV COND RESIDENCIAL ROSA ANDR

In [289]:
df_atividades_economicas[["natureza_juridica", "ind_simples", "ind_mei"]].value_counts().reset_index()\
.assign(ind = lambda df: df["ind_simples"] + df["ind_mei"])\
.assign(soma = lambda df: df.groupby("natureza_juridica")["count"].transform("sum"))\
.pivot_table(index = ["natureza_juridica", "soma"], columns = ["ind"], values = "count")\
.fillna("")\
.reset_index()\
.sort_values("soma", ascending = False)\
.loc[:60]

ind,natureza_juridica,soma,NN,SN,SS
18,EMPRESÁRIO (INDIVIDUAL),578526,13270.0,35484.0,529772.0
48,SOCIEDADE EMPRESÁRIA LIMITADA,309694,139369.0,168129.0,2196.0
7,CONDOMÍNIO EDILÍCIO,26219,26217.0,2.0,
51,SOCIEDADE SIMPLES LIMITADA,20828,11919.0,8909.0,
13,EMPRESA INDIVIDUAL DE RESPONSABILIDADE LIMITAD...,20668,7464.0,13204.0,
43,SOCIEDADE ANÔNIMA FECHADA,12853,12820.0,33.0,
0,ASSOCIAÇÃO PRIVADA,10620,10615.0,5.0,
53,SOCIEDADE UNIPESSOAL DE ADVOGADOS,6037,1887.0,4148.0,2.0
52,SOCIEDADE SIMPLES PURA,5033,2024.0,3009.0,
45,SOCIEDADE EM CONTA DE PARTICIPAÇÃO,3580,3580.0,,


In [267]:
df_atividades_economicas\
.query("~natureza_juridica.str.contains('EMPRES.RIO \(INDIVIDUAL\)')")\
.query("ind_mei == 'S'")

,cnae_principal,descricao_cnae,cnae_secundarias,natureza_juridica,porte_empresa,area_utilizada,ind_simples,ind_mei,tipo_unidade,forma_atuacao,desc_logradouro,nome_logradouro,numero_imovel,complemento,nome_bairro,nome,nome_fantasia,cnpj,data_inicio_atividade,nome_regional,geometria,urlfile,ind_possui_alvara
index,,,,,,,,,,,,,,,,,,,,,,,
110395,4511102.0,"COMERCIO A VAREJO DE AUTOMOVEIS, CAMIONETAS E ...","4511101, 4511103, 4511104, 4511105, 4511106, 4...",SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,"PORTA A PORTA, POSTOS MÓVEIS OU POR AMBULANTES",RUA,DOS AERONAUTAS,256,nan,LIBERDADE,MINUTO VEICULOS LTDA,DOMUS,21986528000151,10/05/2016,PAMPULHA,POINT (609230.784378778 7803808.99547387),20220601_atividade_economica.csv,nan
115525,7319002.0,PROMOCAO DE VENDAS,"4512901, 4930202, 8211300",SOCIEDADE EMPRESÁRIA LIMITADA,EMPRESA DE PEQUENO PORTE,10.0,S,S,UNIDADE PRODUTIVA,"INTERNET, TELEVENDAS",RUA,TELESCOPIO,407,nan,MIRAMAR,DIAS EMPREENDIMENTOS E TRANSPORTES LTDA,DIAS EMPREENDIMENTOS E REPRESENTACAO,26094347000134,02/09/2016,BARREIRO,POINT (603196.638605644 7788604.81976724),20220601_atividade_economica.csv,nan
125852,9602501.0,"CABELEIREIROS, MANICURE E PEDICURE",9602502,SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,PADRE JOSE ALVES,342,nan,SAO PAULO,SALAO JEITO JOVEM LTDA,SALAO JEITO JOVEM,26715726000102,14/12/2016,NORDESTE,POINT (612607.714941593 7802858.48336503),20220601_atividade_economica.csv,nan
172610,6821801.0,CORRETAGEM NA COMPRA E VENDA E AVALIAÇÃO DE IM...,6821802,SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,ATIVIDADES DESENVOLVIDAS FORA DO ESTABELECIMEN...,AVE,ALVARES CABRAL,1162,APT 501,LOURDES,ALENE DUARTE CORRETORA LTDA,REMODELAR ESTETICA,26282896000132,13/06/2017,CENTRO-SUL,POINT (610399.547311648 7795946.96351472),20220601_atividade_economica.csv,nan
181542,5912099.0,"ATIVIDADES DE POS-PRODUCAO CINEMATOGRAFICA, DE...","1813001, 1813099, 1822999, 5811500, 5812301, 5...",SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,"ESTABELECIMENTO FIXO, INTERNET",RUA,CORONEL FULGENCIO,428,APT 6,NOVO SAO LUCAS,PB IDEIAS SOLUCOES EM AUDIOVISUAL LTDA,PB IDEIAS,22796154000174,06/07/2015,CENTRO-SUL,POINT (613412.1168943 7796019.39871637),20220601_atividade_economica.csv,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
993554,1091102.0,FABRICAÇÃO DE PRODUTOS DE PADARIA E CONFEITARI...,"1091102, 4721104, 5611203, 5620104",SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,"INTERNET, PORTA A PORTA, POSTOS MÓVEIS OU POR ...",RUA,PARAIBA,281,nan,FUNCIONARIOS,DOCURAS DA FLAVINHA LTDA,DOCURAS DA FLAVINHA BH,55833484000141,07-07-2024,nan,POINT (611822.67 7796037.99),20251001_atividade_economica.csv,NÃO
993576,2599301.0,SERVIÇOS DE CONFECÇÃO DE ARMAÇÕES METÁLICAS PA...,2599301,SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,EM LOCAL FIXO FORA DE LOJA,RUA,SANTA MARIA,113,nan,VILA NOVA PARAISO,DG ARMACOES EM GERAL LTDA,nan,54816409000100,20-04-2024,nan,POINT (606126.42 7790580.25),20251001_atividade_economica.csv,NÃO
993584,7719599.0,LOCAÇÃO DE OUTROS MEIOS DE TRANSPORTE NÃO ESPE...,7719599,SOCIEDADE EMPRESÁRIA LIMITADA,MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,"CORREIO, EM LOCAL FIXO FORA DE LOJA, ESTABELEC...",RUA,IZABEL ALVES MARTINS,402,nan,SERRANO,JLL LOCACOES DE MEIOS DE TRANSPORTE LTDA,JLL LOCACOES,55167984000191,17-05-2024,nan,POINT (603548.77 7801340.83),20251001_atividade_economica.csv,NÃO


In [178]:
xdf.query("tp_ == 'SS0'")

,cnpj,nome,nome_fantasia,data_inicio_atividade,nome_regional,cnae_principal,cnae_secundarias,descricao_cnae,porte_empresa,area_utilizada,ind_simples,ind_mei,tipo_unidade,forma_atuacao,desc_logradouro,nome_logradouro,numero_imovel,complemento,nome_bairro,geometria,ind_id,tp_nome,tp_
index,,,,,,,,,,,,,,,,,,,,,,,


In [179]:
df_empresarios_individuais\
.query("ind_id == 'SS0'")

,cnpj,nome,nome_fantasia,data_inicio_atividade,nome_regional,cnae_principal,cnae_secundarias,descricao_cnae,porte_empresa,area_utilizada,ind_simples,ind_mei,tipo_unidade,forma_atuacao,desc_logradouro,nome_logradouro,numero_imovel,complemento,nome_bairro,geometria,ind_id
index,,,,,,,,,,,,,,,,,,,,,
55,71046767000108,HELIO JOSE DOS PASSOS,nan,01/06/1993,OESTE,9521500.0,nan,REPARACAO E MANUTENCAO DE EQUIPAMENTOS ELETROE...,MICROEMPRESA - ME,69.0,S,S,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,AVE,TERESA CRISTINA,8000,nan,BAIRRO DAS INDUSTRIAS II,POINT (604832.714235935 7792733.44122486),SS0
77,70985205000159,ADEMIR SANTANA DA SILVA,CICE BALAS,01/04/1993,BARREIRO,4712100.0,"4713002, 4721104, 4789099","COMERCIO VAREJISTA DE MERCADORIAS EM GERAL, CO...",MICROEMPRESA - ME,113.0,S,S,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,ERIDANO,245,LOJA 2,CARDOSO,POINT (603663.25825095 7788538.41657597),SS0
188,23962111000184,ANA R L BRAGA BITENCOURT CONFECCOES,BRAGA & BITENCOURT INDUSTRIA E COMERCIO DE CON...,01/03/1993,BARREIRO,1412602.0,1412603,"CONFECCAO, SOB MEDIDA, DE PECAS DO VESTUARIO, ...",MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,CASTOR,147,SALA 01,MIRAMAR,POINT (603381.602157723 7788738.6632132),SS0
304,71077689000109,ELVECIO ALVES DE OLIVEIRA,CHAVEIRO GUARARAPES,01/06/1993,NOROESTE,9529102.0,8299703,CHAVEIROS,DEMAIS,20.0,S,S,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,GUARARAPES,1300,C,PINDORAMA,POINT (602776.994906018 7798403.63853863),SS0
372,65096224000194,ROSANGELA FERREIRA CAMPOS MACHADO - CALCADOS,SCARPE MODAS,18/02/1991,PAMPULHA,4782201.0,"4755502, 4781400, 4789099",COMERCIO VAREJISTA DE CALCADOS,MICROEMPRESA - ME,37.0,S,S,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,AVE,CORONEL JOSE DIAS BICALHO,1221,nan,SAO JOSE,POINT (607782.12181818 7803582.56055663),SS0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
881290,28900665000170,M. A. DA SILVA FIRMINO,VISAO DE AGUIA COLCHOES,20-10-2017,nan,4754702.0,"4754701, 4754702, 4759899, 4781400, 4783101, 4...",COMERCIO VAREJISTA DE ARTIGOS DE COLCHOARIA,MICROEMPRESA - ME,18.0,S,S,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,RADIALISTA JUSCELINO SOUZA,310,nan,CEU AZUL,POINT (603987.99 7808607.97),SS0
893024,17020087000161,JOSE CARLOS RODRIGUES DOS SANTOS,nan,17-10-2012,nan,4520006.0,4520006,SERVICOS DE BORRACHARIA PARA VEICULOS AUTOMOTORES,MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,RIBATEJO,110,nan,CACHOEIRINHA,POINT (609880.35 7801820.00),SS0
898729,31141084000107,DENILSON DE PAULA SOLUCOES EM EQUIPAMENTOS REC...,MVL SOLUCOES,07-08-2018,nan,3314710.0,"3314710, 7319003, 7721700",MANUTENCAO E REPARACAO DE MAQUINAS E EQUIPAMEN...,MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,MÁQUINAS AUTOMÁTICAS,RUA,INDIRA GANDHI,133,nan,TUPI A,POINT (613024.10 7806591.46),SS0


In [133]:
xdf.query("tp_ == 0")

,cnpj,nome,nome_fantasia,data_inicio_atividade,nome_regional,cnae_principal,cnae_secundarias,descricao_cnae,porte_empresa,area_utilizada,ind_simples,ind_mei,tipo_unidade,forma_atuacao,desc_logradouro,nome_logradouro,numero_imovel,complemento,nome_bairro,geometria,ind_id,tp_nome,tp_
index,,,,,,,,,,,,,,,,,,,,,,,


In [46]:
df_atividades_economicas\
.query("natureza_juridica.str.contains('EMPRES.RIO \(INDIVIDUAL\)')")

,cnae_principal,descricao_cnae,cnae_secundarias,natureza_juridica,porte_empresa,area_utilizada,ind_simples,ind_mei,tipo_unidade,forma_atuacao,desc_logradouro,nome_logradouro,numero_imovel,complemento,nome_bairro,nome,nome_fantasia,cnpj,data_inicio_atividade,nome_regional,geometria,urlfile,ind_possui_alvara
index,,,,,,,,,,,,,,,,,,,,,,,
27,7500100.0,ATIVIDADES VETERINÁRIAS EXERCIDAS EM CLÍNICAS ...,"4771704, 4789004, 4789099, 9609207, 9609208",EMPRESÁRIO (INDIVIDUAL),MICROEMPRESA - ME,55.0,S,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,AVE,ARTUR GUIMARAES,155,nan,SANTA CRUZ,MARCIO BRASIL ROCHA,nan,66477498000196,22/03/1993,NORDESTE,POINT (611354.775664121 7801545.91893658),20220601_atividade_economica.csv,nan
33,4724500.0,COMERCIO VAREJISTA DE HORTIFRUTIGRANJEIROS,"4712100, 5611203",EMPRESÁRIO (INDIVIDUAL),MICROEMPRESA - ME,81.0,S,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,CIBIPURUNA,171,nan,MONTE AZUL,MANOEL ADUIRES DA SILVA,MANOEL ADUIRES DA SILVA,38462867000131,01/04/1990,NORTE,POINT (615915.892278689 7808531.04864468),20220601_atividade_economica.csv,nan
43,4520001.0,SERVICOS DE MANUTENCAO E REPARACAO MECANICA DE...,"4520002, 4520003, 4520004",EMPRESÁRIO (INDIVIDUAL),MICROEMPRESA - ME,150.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,DOUTOR BENJAMIM MOSS,319,nan,UNIAO,ROBSON SALES HENEDINO,nan,70986948000143,01/04/1993,NORDESTE,POINT (613033.200778877 7800638.00374008),20220601_atividade_economica.csv,nan
47,4772500.0,"COMERCIO VAREJISTA DE COSMETICOS, PRODUTOS DE ...","4721104, 4723700, 4789099, 5611203",EMPRESÁRIO (INDIVIDUAL),MICROEMPRESA - ME,20.0,S,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,JOSE BRANDAO,174,LOJA,BARREIRO,PAUL BLAHA,nan,68553478000182,01/12/1992,BARREIRO,POINT (603131.422191837 7790981.54869036),20220601_atividade_economica.csv,nan
55,9521500.0,REPARACAO E MANUTENCAO DE EQUIPAMENTOS ELETROE...,nan,EMPRESÁRIO (INDIVIDUAL),MICROEMPRESA - ME,69.0,S,S,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,AVE,TERESA CRISTINA,8000,nan,BAIRRO DAS INDUSTRIAS II,HELIO JOSE DOS PASSOS,nan,71046767000108,01/06/1993,OESTE,POINT (604832.714235935 7792733.44122486),20220601_atividade_economica.csv,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1536710,4781400.0,COMERCIO VAREJISTA DE ARTIGOS DO VESTUARIO E C...,"1412602, 4781400",EMPRESÁRIO (INDIVIDUAL),MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,CASTIGLIANO,621,APT 104,PADRE EUSTAQUIO,36.766.918 JOANA ANGELICA SILVA DE SOUZA,JOANA ANGELICA EMPREEDIMENTOS,36766918000193,25-03-2020,nan,POINT (607217.55 7797321.52),20251103_atividade_economica.csv,NÃO
1536924,5620104.0,FORNECIMENTO DE ALIMENTOS PREPARADOS PREPONDER...,"5611203, 5620104, 9602501, 9602502",EMPRESÁRIO (INDIVIDUAL),MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,"ESTABELECIMENTO FIXO, INTERNET",RUA,L,100,LOJA LOJA,CONJUNTO MINASCAIXA,37.087.365 ELISABETH FERNANDA GARCIA CARDOSO B...,LANCHES EXPRESS,37087365000105,07-05-2020,nan,POINT (609085.97 7809993.00),20251103_atividade_economica.csv,NÃO
1537457,4930202.0,"TRANSPORTE RODOVIARIO DE CARGA, EXCETO PRODUTO...","4930201, 4930202, 4930204",EMPRESÁRIO (INDIVIDUAL),MICROEMPRESA - ME,10.0,S,S,UNIDADE PRODUTIVA,"EM LOCAL FIXO FORA DE LOJA, ESTABELECIMENTO FI...",RUA,O ATENEU,19,CASA CS,ADEMAR MALDONADO,28.169.364 ANDREIA RODRIGUES CLARK CAMPOS,nan,28169364000119,27-01-2025,nan,POINT (601943.94 7790116.07),20251103_atividade_economica.csv,NÃO


In [23]:
df_atividades_economicas["natureza_juridica"].value_counts()

natureza_juridica
EMPRESÁRIO (INDIVIDUAL)                                                     578526
SOCIEDADE EMPRESÁRIA LIMITADA                                               309694
CONDOMÍNIO EDILÍCIO                                                          26219
SOCIEDADE SIMPLES LIMITADA                                                   20828
EMPRESA INDIVIDUAL DE RESPONSABILIDADE LIMITADA (DE NATUREZA EMPRESARIA)     20668
                                                                             ...  
CANDIDATO A CARGO POLÍTICO ELETIVO                                               1
COMISSÃO DE CONCILIAÇÃO PRÉVIA                                                   1
COMUNIDADE INDÍGENA                                                              1
GRUPO DE SOCIEDADES                                                              1
FUNDAÇÃO PÚBLICA DE DIREITO PRIVADO ESTADUAL OU DO DISTRITO FEDERAL              1
Name: count, Length: 68, dtype: int64

In [22]:
df_atividades_economicas["porte_empresa"].value_counts()

porte_empresa
MICROEMPRESA - ME           825571
DEMAIS                      126369
EMPRESA DE PEQUENO PORTE     54749
Name: count, dtype: int64

In [19]:
df_atividades_economicas.query("cnpj == '51427102000552'")

,cnae_principal,descricao_cnae,cnae_secundarias,natureza_juridica,porte_empresa,area_utilizada,ind_simples,ind_mei,tipo_unidade,forma_atuacao,desc_logradouro,nome_logradouro,numero_imovel,complemento,nome_bairro,nome,nome_fantasia,cnpj,data_inicio_atividade,nome_regional,geometria,urlfile,ind_possui_alvara
index,,,,,,,,,,,,,,,,,,,,,,,
51981,6619304.0,CAIXAS ELETRONICOS,nan,SOCIEDADE ANÔNIMA FECHADA,DEMAIS,2.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,PROFESSOR ESTEVAO PINTO,830,nan,SERRA,TECNOLOGIA BANCARIA S.A,nan,51427102000552,18/05/2001,CENTRO-SUL,POINT (612812.8219499 7794497.91036454),20220601_atividade_economica.csv,nan
51982,6619304.0,CAIXAS ELETRONICOS,nan,SOCIEDADE ANÔNIMA FECHADA,DEMAIS,2.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,AVE,DO CONTORNO,6619,nan,SANTO ANTONIO,TECNOLOGIA BANCARIA S.A,nan,51427102000552,04/01/2001,CENTRO-SUL,POINT (610990.39807364 7794799.17092164),20220601_atividade_economica.csv,nan
51983,6619304.0,CAIXAS ELETRONICOS,nan,SOCIEDADE ANÔNIMA FECHADA,DEMAIS,1.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,MARIA AMELIA MAIA,811,nan,SAO BERNARDO,TECNOLOGIA BANCARIA S.A,nan,51427102000552,02/02/2000,NORTE,POINT (610697.884926454 7805264.82673433),20220601_atividade_economica.csv,nan
51984,6619304.0,CAIXAS ELETRONICOS,nan,SOCIEDADE ANÔNIMA FECHADA,DEMAIS,1.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,RUA,ERICO VERISSIMO,1685,nan,SANTA MONICA,TECNOLOGIA BANCARIA S.A,nan,51427102000552,04/01/2001,VENDA NOVA,POINT (607169.073629446 7808424.30107708),20220601_atividade_economica.csv,nan
51985,6619304.0,CAIXAS ELETRONICOS,nan,SOCIEDADE ANÔNIMA FECHADA,DEMAIS,2.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,AVE,CRISTIANO MACHADO,2250,nan,CIDADE NOVA,TECNOLOGIA BANCARIA S.A,nan,51427102000552,03/11/2000,NORDESTE,POINT (612125.272379601 7800604.93007157),20220601_atividade_economica.csv,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
987623,6311900.0,"TRATAMENTO DE DADOS, PROVEDORES DE SERVIÇOS DE...",6311900,SOCIEDADE ANÔNIMA FECHADA,DEMAIS,1.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,BEC,RAUL SOARES,70,nan,VILA ANTENA,TECNOLOGIA BANCARIA S.A.,nan,51427102000552,07-08-2025,nan,POINT (608515.72 7793925.83),20251001_atividade_economica.csv,NÃO
987624,6311900.0,"TRATAMENTO DE DADOS, PROVEDORES DE SERVIÇOS DE...",6311900,SOCIEDADE ANÔNIMA FECHADA,DEMAIS,1.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,PCA,RUI BARBOSA,50,"NÃ?MERO:0,ANEXO:PC 56532",CENTRO,TECNOLOGIA BANCARIA S.A.,nan,51427102000552,07-08-2025,nan,POINT (611594.68 7797470.97),20251001_atividade_economica.csv,NÃO
987625,6311900.0,"TRATAMENTO DE DADOS, PROVEDORES DE SERVIÇOS DE...",6311900,SOCIEDADE ANÔNIMA FECHADA,DEMAIS,1.0,N,N,UNIDADE PRODUTIVA,ESTABELECIMENTO FIXO,PCA,RUI BARBOSA,50,"NÃ?MERO:0,ANEXO:PC 56556",CENTRO,TECNOLOGIA BANCARIA S.A.,nan,51427102000552,07-08-2025,nan,POINT (611594.68 7797470.97),20251001_atividade_economica.csv,NÃO
